# Day 7 — Structured output: JSON you can trust

**Phase 1 · Generative AI Foundations · ~40 min · Needs a provider**

## 🎯 Objective
Get **reliable JSON** out of a model and validate it — the skill that lets GenAI
feed *other code* (and the foundation of agents choosing tools).

## 🧠 Concept
Models are chatty: they wrap JSON in code fences or add prose. The professional
pattern is **generate → validate → repair**:

```mermaid
flowchart LR
    P["Prompt: 'reply with ONLY JSON'"] --> G["🧠 Generate"]
    G --> V{"Valid?<br/>parse + schema"}
    V -- yes --> Done["✅ Use it"]
    V -- no --> R["🔧 Send error back, retry"] --> V
```

We validate with **Pydantic**: define the shape once; it parses and type-checks,
raising a clear error the model can use to self-correct.

## 🔬 Exercise
Implement `extract_task()` (prompt for JSON) and `parse_or_repair()` (parse +
validate + retry on failure).

## ✅ Done when
- You get a typed `Task` back even when the model is initially chatty.

## 📝 Reflect
1. Why is sending the *validation error* back to the model so effective?
2. How will this exact skill power an agent that must choose a tool + arguments?

## 🔭 Tomorrow
Day 8: give meaning to text with **embeddings** and similarity.


### ▶ Run this first

In [ ]:
# Run me first: make the course's shared/ package importable from this notebook.
import sys, pathlib
for _cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_cand / 'shared' / 'llm.py').exists():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break


## 🔬 Your turn
Complete the `TODO`s below, then run the cell (**Shift+Enter**).

In [ ]:
import json

from pydantic import BaseModel, Field, ValidationError

from shared.llm import chat


class Task(BaseModel):
    title: str
    priority: int = Field(ge=1, le=5)
    done: bool


def _strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if "\n" in s:
            s = s.split("\n", 1)[1]
    return s.strip()


def extract_task(text):
    """Prompt for JSON (title, priority 1-5, done) then call parse_or_repair.

    TODO: raw = chat([system, user=text], temperature=0); return parse_or_repair(raw)
    """
    raise NotImplementedError


def parse_or_repair(raw, attempts=2):
    """Parse JSON -> Task; on failure, ask the model to fix it (<= attempts).

    TODO: loop: try json.loads(_strip_fences(raw)) -> Task(**data);
          except (JSONDecodeError, ValidationError) as e: raw = chat([...send error...])
    """
    raise NotImplementedError


if __name__ == "__main__":
    print(extract_task("remind me to call the dentist, pretty urgent"))


---
---
## 🔒 Solution
*Try it yourself first!* Run the cell below only when you're ready to compare.

In [ ]:
import json

from pydantic import BaseModel, Field, ValidationError

from shared.llm import chat


class Task(BaseModel):
    title: str
    priority: int = Field(ge=1, le=5)
    done: bool


def _strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
        if "\n" in s:
            s = s.split("\n", 1)[1]
    return s.strip()


def extract_task(text):
    raw = chat(
        [
            {"role": "system", "content":
             "Extract a task as JSON with keys title (string), priority (integer 1-5), "
             "done (boolean). Reply with ONLY JSON."},
            {"role": "user", "content": text},
        ],
        temperature=0,
    )
    return parse_or_repair(raw)


def parse_or_repair(raw, attempts=2):
    for _ in range(attempts + 1):
        try:
            return Task(**json.loads(_strip_fences(raw)))
        except (json.JSONDecodeError, ValidationError) as exc:
            raw = chat(
                [
                    {"role": "system", "content":
                     "Fix this into valid JSON with keys title (string), priority "
                     "(integer 1-5), done (boolean). ONLY JSON."},
                    {"role": "user", "content": f"Error: {exc}\nBroken: {raw}"},
                ],
                temperature=0,
            )
    raise ValueError("Could not produce valid JSON.")


if __name__ == "__main__":
    print(extract_task("remind me to call the dentist, pretty urgent"))
